<a href="https://colab.research.google.com/github/gratefulgee/fcc-ML-NN-sms-text-classifier/blob/main/copy_of_fcc_sms_text_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# import libraries
!pip install tensorflow
import tensorflow as tf
import pandas as pd
from tensorflow import keras
!pip install tensorflow-datasets
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt

print(tf.__version__)

In [ ]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/sms/train-data.tsv
!wget https://cdn.freecodecamp.org/project-data/sms/valid-data.tsv

train_file_path = "train-data.tsv"
test_file_path = "valid-data.tsv"

In [ ]:
train = pd.read_csv("train-data.tsv", sep="\t", header=None, names=["label", "message"])
test  = pd.read_csv("valid-data.tsv", sep="\t", header=None, names=["label", "message"])
train.head(100)

In [ ]:
# convert label
train["label_n"] = train["label"].apply(lambda x: 1 if x=="spam" else 0)
test["label_n"] = test["label"].apply(lambda x: 1 if x=="spam" else 0)

#convert to numpy array
train_text = train["message"].astype(str).values
train_label = train["label_n"].values

test_text = test["message"].astype(str).values
test_label = test["label_n"].values
train_text

In [ ]:
# Tokenizer
tokenizer = tf.keras.preprocessing.text.Tokenizer(num_words=20000, oov_token="<OOV>")
tokenizer.fit_on_texts(train_text)

# convert to sequence
train_seq = tokenizer.texts_to_sequences(train_text)
test_seq = tokenizer.texts_to_sequences(test_text)

# padding
max_len = 100
train_pad = tf.keras.preprocessing.sequence.pad_sequences(train_seq, maxlen=max_len, padding='post')
test_pad = tf.keras.preprocessing.sequence.pad_sequences(test_seq, maxlen=max_len, padding='post')


In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(20000, 32, input_length=max_len),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(32, return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(16)),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.fit(train_pad, train_label, epochs=5, validation_split=0.2)

loss, acc = model.evaluate(test_pad, test_label)
print("ACC =", acc)

In [ ]:
# function to predict messages based on model
# (should return list containing prediction and label, ex. [0.008318834938108921, 'ham'])
def predict_message(pred_text):
    seq = tokenizer.texts_to_sequences([pred_text])
    pad = tf.keras.preprocessing.sequence.pad_sequences(seq, maxlen=max_len, padding='post')
    prob = model.predict(pad)[0][0]
    label = 'spam' if prob > 0.5 else 'ham'
    return [float(prob), label]

pred_text = "how are you doing today?"

prediction = predict_message(pred_text)
print(prediction)

In [ ]:
# Run this cell to test your function and model. Do not modify contents.
def test_predictions():
  test_messages = ["how are you doing today",
                   "sale today! to stop texts call 98912460324",
                   "i dont want to go. can we try it a different day? available sat",
                   "our new mobile video service is live. just install on your phone to start watching.",
                   "you have won £1000 cash! call to claim your prize.",
                   "i'll bring it tomorrow. don't forget the milk.",
                   "wow, is your arm alright. that happened to me one time too"
                  ]

  test_answers = ["ham", "spam", "ham", "spam", "spam", "ham", "ham"]
  passed = True

  for msg, ans in zip(test_messages, test_answers):
    prediction = predict_message(msg)
    if prediction[1] != ans:
      passed = False

  if passed:
    print("You passed the challenge. Great job!")
  else:
    print("You haven't passed yet. Keep trying.")

test_predictions()
